<a href="https://colab.research.google.com/github/jennifer-algabre/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!git clone https://github.com/jennifer-algabre/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 128 (delta 43), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.85 MiB | 9.73 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/flyrank-ml-internship/flyrank-ml-internship


In [7]:
!find . -name "*.csv"

./outputs/refresh_queue_sample.csv
./data/raw/content_refresh_anonymized.csv


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector consists of observable search performance, engagement, content freshness, and content characteristics that are available before making a content refresh decision.

Missing numerical values are filled with 0 for this starter notebook, while categorical variables are encoded using one-hot encoding. The resulting feature vector is intended for decision support rather than production use.

In [8]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "content_type",
    "main_intent"
]

X = df[feature_cols].copy()

# Fill missing numerical values
numeric_cols = X.select_dtypes(include="number").columns
X[numeric_cols] = X[numeric_cols].fillna(0)

# One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=["content_type", "main_intent"],
    drop_first=True
)

print("Feature vector shape:", X.shape)
X.head()

Feature vector shape: (30000, 16)


,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,scroll_rate,word_count,content_age_days,days_since_last_update,search_volume,competition,content_type_feedly article,content_type_keyword article,main_intent_informational,main_intent_navigational,main_intent_transactional
0,3803,29,0.76,10.6,5.88,4.55,3221.0,187,20,10.0,0.67,False,True,False,False,True
1,15320,7,0.05,20.3,0.00,10.00,2481.0,445,25,90.0,0.01,False,True,True,False,False
2,12581,11,0.09,36.5,0.00,28.57,3515.0,141,20,0.0,0.00,False,True,True,False,False
3,11751,58,0.49,6.2,1.28,3.45,0.0,463,22,10.0,0.00,False,True,False,False,False
4,19140,24,0.13,44.0,0.00,24.29,2803.0,263,14,0.0,0.00,False,True,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing values | Available before prediction? |
|---------|---------|---------------|------------------------------|
| impressions_90d | Search impressions over the previous 90 days | Filled with 0 | Yes |
| clicks_90d | Search clicks over the previous 90 days | Filled with 0 | Yes |
| ctr | Click-through rate | Filled with 0 | Yes |
| avg_position | Average search ranking position | Filled with 0 | Yes |
| engagement_rate | User engagement metric | Filled with 0 | Yes |
| scroll_rate | User scroll behavior | Filled with 0 | Yes |
| word_count | Number of words in the content | Filled with 0 | Yes |
| content_age_days | Age of the content | Filled with 0 | Yes |
| days_since_last_update | Days since last update | Filled with 0 | Yes |
| search_volume | Estimated keyword search volume | Filled with 0 | Yes |
| competition | Keyword competition score | Filled with 0 | Yes |
| content_type | Content category (one-hot encoded) | Encoded | Yes |
| main_intent | Search intent category (one-hot encoded) | Encoded | Yes |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The feature vector was checked for columns that could leak information about the prediction target.

The target for this lane is based on trend direction, so fields directly derived from trend outcomes were excluded. Product decision flags and future information should also not be used because they would not be available at prediction time.

The following code lists potential leakage columns found in the starter dataset.

In [9]:
potential_leakage = [
    col for col in df.columns
    if "trend" in col.lower()
    or "label" in col.lower()
    or "score" in col.lower()
]

print("Potential leakage columns:")
print(potential_leakage)

Potential leakage columns:
['trend_direction', 'trend_pct']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Reason |
|---------------|--------|
| trend_direction | Used as the prediction target and cannot be used as a feature. |
| trend_pct | Directly related to the target and would cause label leakage. |
| content_id | Identifier only; not predictive. |
| client_id | Identifier only; may encourage memorization instead of learning general patterns. |
| ai_traffic_pct | Not required for this baseline feature vector and may not consistently represent observable search behavior. |

In [10]:
excluded = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "ai_traffic_pct"
]

print("Excluded features:")
for feature in excluded:
    print("-", feature)

Excluded features:
- trend_direction
- trend_pct
- content_id
- client_id
- ai_traffic_pct
